In [1]:
import json, re, time, os
from openai import OpenAI

POE_API_KEY = "sk-poe-4MPbglPbB-R8NF-J-7UcRmP4OtJUM0roCRgSMkAI54I" # This API Key is deactivated so it's fine to expose it
MODEL_NAME = "deepseek-v4.1-flash-1017"

client = OpenAI(api_key=POE_API_KEY, base_url="https://api.poe.com/v1")

# 1. Load updated cc-site-info.json and richku-info.json
with open("cc-site-info.json", "r") as f:
    cc_data = json.load(f)

ric_file = "richku-info.json" if os.path.exists("richku-info.json") else "ric-info.json"
with open(ric_file, "r") as f:
    ric_data = json.load(f)

# 2. Load EXISTING llm_scores so we don't re-run already scored courses!
if os.path.exists("ric_llm_scores.json"):
    with open("ric_llm_scores.json", "r") as f:
        llm_scores = json.load(f)
else:
    llm_scores = {}

active_codes = sorted([c.strip().upper() for c in cc_data.keys() if not c.strip().upper().startswith("CCAI")])

# Find ONLY newly added courses that aren't in llm_scores yet
new_courses_to_score = {}
for code in active_codes:
    if code in llm_scores:
        continue # Already scored!
    c_data = ric_data.get(code, {})
    reviews = c_data.get("reviews", [])
    texts = [(r.get("translated_content") or r.get("content") or "").strip() for r in reviews]
    texts = [t for t in texts if t]
    if texts:
        new_courses_to_score[code] = texts

print(f"Already scored: {len(llm_scores)} courses.")
print(f"Newly added courses needing LLM scoring: {len(new_courses_to_score)}")

for idx, (code, reviews_list) in enumerate(new_courses_to_score.items()):
    formatted_reviews = "\n".join([f"- Review {i+1}: {rev}" for i, rev in enumerate(reviews_list)])
    prompt = f"""You are evaluating student reviews for the HKU Common Core course {code}.
Read the following {len(reviews_list)} student reviews and output two continuous scores (from 1.0 to 5.0, rounded to 2 decimal places):
1. "llm_sentiment_score": Overall student experience/sentiment from 1.0 (terrible experience, unfair grading, boring) to 5.0 (amazing experience, interesting, great teaching/grading).
2. "llm_workload_score": Perceived course workload from 1.0 (very chill, light workload, easy) to 5.0 (very heavy workload, stressful, demanding).

Student Reviews for {code}:
{formatted_reviews}

Respond ONLY with a valid JSON object in this exact format, with no extra text or markdown:
{{"llm_sentiment_score": 3.50, "llm_workload_score": 2.80}}"""

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": "You are a precise data science assistant that outputs only valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0
            )
            raw_reply = response.choices[0].message.content.strip()
            json_match = re.search(r"\{[^{}]*\"llm_sentiment_score\"[^{}]*\}", raw_reply, re.DOTALL)
            parsed = json.loads(json_match.group(0))
            llm_scores[code] = {
                "llm_sentiment_score": max(1.0, min(5.0, round(float(parsed["llm_sentiment_score"]), 2))),
                "llm_workload_score": max(1.0, min(5.0, round(float(parsed["llm_workload_score"]), 2))),
                "ric_review_count": len(reviews_list)
            }
            print(f"[{idx+1}/{len(new_courses_to_score)}] Scored {code}: {llm_scores[code]}")
            break
        except Exception as e:
            print(f"Error on {code}: {e}")
            time.sleep(2)

with open("ric_llm_scores.json", "w") as f:
    json.dump(llm_scores, f, indent=4)
print(f"Total courses in ric_llm_scores.json: {len(llm_scores)}")

KeyboardInterrupt: 

In [5]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://sweb.hku.hk/ccacad/ccc_appl/enrol_stat.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

enrol_data = {}
current_sem = "sem1"

# Parse every table row in order
for tr in soup.find_all("tr"):
    row_text = tr.get_text(" ", strip=True)
    if "First Semester" in row_text:
        current_sem = "sem1"
        continue
    elif "Second Semester" in row_text:
        current_sem = "sem2"
        continue

    tds = [td.get_text(strip=True) for td in tr.find_all("td")]
    if len(tds) >= 5 and re.match(r"^CC[A-Z]{2}\d{4}$", tds[0].upper()):
        code = tds[0].strip().upper()
        subclass = tds[2].strip().upper()

        # Skip CCAI and skip Exchange subclasses (AX, BX, CX, DX)
        if code.startswith("CCAI") or subclass.endswith("X"):
            continue

        try:
            quota = int(tds[3])
            vacancies = int(tds[4])
        except ValueError:
            continue

        if code not in enrol_data:
            enrol_data[code] = {
                "sem1_quota": 0, "sem1_vacancies": 0,
                "sem2_quota": 0, "sem2_vacancies": 0,
                "regular_subclasses": []
            }

        enrol_data[code][f"{current_sem}_quota"] += quota
        enrol_data[code][f"{current_sem}_vacancies"] += vacancies
        enrol_data[code]["regular_subclasses"].append(f"{current_sem}_{subclass}")

with open("enrol-stat-info.json", "w") as f:
    json.dump(enrol_data, f, indent=4)

print(f"Successfully scraped {len(enrol_data)} active non-CCAI courses into 'enrol-stat-info.json'!")
# Preview 3 courses to confirm sem1_quota and sem2_quota are non-zero:
for k in list(enrol_data.keys())[:3]:
    print(k, enrol_data[k])

Successfully scraped 171 active non-CCAI courses into 'enrol-stat-info.json'!
CCST5037 {'sem1_quota': 120, 'sem1_vacancies': 1, 'sem2_quota': 0, 'sem2_vacancies': 0, 'regular_subclasses': ['sem1_A']}
CCST9003 {'sem1_quota': 120, 'sem1_vacancies': 1, 'sem2_quota': 0, 'sem2_vacancies': 0, 'regular_subclasses': ['sem1_A']}
CCST9006 {'sem1_quota': 120, 'sem1_vacancies': 0, 'sem2_quota': 0, 'sem2_vacancies': 0, 'regular_subclasses': ['sem1_A']}


In [ ]:
import json
import pandas as pd
import numpy as np
import re

# 1. Load all JSON files
with open("cc-site-info.json", "r") as f:
    cc_raw = json.load(f)

with open("sis-info.json", "r") as f:
    sis_raw = json.load(f)

with open("enrol-stat-info.json", "r") as f:
    enrol_raw = json.load(f)

try:
    with open("ric_llm_scores.json", "r") as f:
        llm_raw = json.load(f)
except FileNotFoundError:
    llm_raw = {}

# ---------------------------------------------------------
# 2. Upgraded Rule-Based Categorization Functions (0 Unmatched!)
# ---------------------------------------------------------
unmatched_load_keys = set()
unmatched_ass_keys = set()

def categorize_study_load(load_dict):
    """
    Maps study_load_dist into:
    hours_lectures, hours_tutorials, hours_self_study, hours_assessment
    """
    buckets = {
        "hours_lectures": 0.0,
        "hours_tutorials": 0.0,
        "hours_self_study": 0.0,
        "hours_assessment": 0.0
    }
    for raw_k, raw_v in load_dict.items():
        k = raw_k.strip().lower()
        val = float(str(raw_v).strip())

        # Rule 1: Assessment preparation & tasks
        if k.startswith("assessment") or any(w in k for w in [
            "essay", "report", "presentation", "project", "quiz", "test", "exam",
            "assignment", "writing", "journal", "paper", "portfolio", "poster",
            "debate", "podcast", "video", "blog", "critique", "reflection",
            "preparation for", "assessment task", "group work"
        ]):
            buckets["hours_assessment"] += val
        # Rule 2: Lectures / Seminars / Classroom sessions
        elif any(w in k for w in ["lecture", "seminar", "classroom session"]):
            buckets["hours_lectures"] += val
        # Rule 3: Reading / Self-study / Online material / Research / Movie viewing
        elif any(w in k for w in [
            "reading", "self-study", "self study", "online learning", "independent study",
            "revision", "independent research", "fact-finding", "case-based inquiry",
            "internet search", "internet research", "online course material",
            "studying movies", "movie viewing", "case studies"
        ]):
            buckets["hours_self_study"] += val
        # Rule 4: Interactive / Experiential classes (Tutorials, Labs, Practicals, PBL, Games, Out of class)
        elif any(w in k for w in [
            "tutorial", "field", "visit", "trip", "workshop", "lab", "film",
            "screening", "discussion", "practicum", "studio", "excursion",
            "activity", "consultation", "practical", "problem-based learning",
            "interactive game", "online participation", "demonstration",
            "out of class experience"
        ]):
            buckets["hours_tutorials"] += val
        else:
            unmatched_load_keys.add(raw_k)
            buckets["hours_self_study"] += val

    return buckets

def categorize_assessment(ass_dict):
    """
    Maps ass_dist into:
    weight_essay, weight_quiz_exam, weight_presentation, weight_participation, weight_project
    Splits compound tasks evenly across matched categories and normalizes to 100%.
    """
    buckets = {
        "weight_essay": 0.0,
        "weight_quiz_exam": 0.0,
        "weight_presentation": 0.0,
        "weight_participation": 0.0,
        "weight_project": 0.0
    }

    for raw_k, raw_v in ass_dict.items():
        k = raw_k.strip().lower()
        val = float(str(raw_v).strip().replace("%", "").strip())

        matched_cats = []

        # 1. Check Essay / Written / Case & Film & Data Analysis / Research Proposal / Notes
        if any(w in k for w in [
            "essay", "writing", "written", "report", "journal", "paper",
            "reflection", "reflective", "review", "critique", "commentary",
            "diary", "blog", "response", "summary", "editorial", "opinion", "article",
            "case file", "film analysis", "case analysis", "data analysis",
            "observation notes", "research proposal", "research"
        ]):
            matched_cats.append("weight_essay")

        # 2. Check Quiz / Test / Exam / Problem sets / Study questions
        if any(w in k for w in [
            "quiz", "test", "exam", "mcq", "multiple choice", "multiple-choice",
            "mid-term", "midterm", "weekly study questions", "problem sets"
        ]):
            matched_cats.append("weight_quiz_exam")

        # 3. Check Presentation / Oral / Debate / Interview
        if any(w in k for w in [
            "presentation", "oral", "debate", "pitch", "speech", "symposium",
            "panel", "role-play", "role play", "interview"
        ]):
            matched_cats.append("weight_presentation")

        # 4. Check Participation / Attendance / Tutorials / Labs / PBL / Field trip / Group activity
        if any(w in k for w in [
            "participation", "attendance", "discussion", "in-class", "in class",
            "tutorial exercise", "tutorial task", "tutorial work", "tutorial activity",
            "classwork", "class exercise", "engagement", "forum", "peer", "homework",
            "worksheet", "problem-based learning", "laboratory", "practical experiment",
            "field trip", "group activity", "film viewing and internet search"
        ]):
            matched_cats.append("weight_participation")

        # 5. Check Project / Creative / Film / Simulation / Infographic / VR / Elective
        if any(w in k for w in [
            "project", "poster", "podcast", "video", "multimedia", "portfolio",
            "exhibition", "creative", "design", "prototype", "fieldwork", "case study",
            "dossier", "simulation", "film production", "instant movies",
            "infographic", "virtual reality", "film festival", "3-minute film",
            "elective item"
        ]):
            matched_cats.append("weight_project")

        # Fallback for generic words
        if not matched_cats:
            if "tutorial" in k or "lecture" in k:
                matched_cats.append("weight_participation")
            elif "assignment" in k or "work" in k or "task" in k:
                matched_cats.append("weight_essay")
            elif "assessment" in k:
                matched_cats.append("weight_quiz_exam")
            else:
                unmatched_ass_keys.add(raw_k)
                matched_cats.append("weight_project")

        split_val = val / len(matched_cats)
        for cat in matched_cats:
            buckets[cat] += split_val

    total_w = sum(buckets.values())
    if total_w > 0:
        for cat in buckets:
            buckets[cat] = round((buckets[cat] / total_w) * 100.0, 2)

    return buckets

# ---------------------------------------------------------
# 3. Pre-aggregate SIS Dataset (Response-Weighted Average)
# ---------------------------------------------------------
sis_eff_clean = {}
sis_enrol_clean = {}
sis_quota_est = {}
for code, records in sis_raw.items():
    code = code.strip().upper()
    if code.startswith("CCAI"):
        continue
    valid_records = []
    max_enrol = 0
    for r in records:
        try:
            enrol = float(r.get("enrollment", 0))
            resp = float(r.get("responses", 0))
            eff = float(r.get("mean_course_effective", np.nan))
            max_enrol = max(max_enrol, enrol)
            if enrol > 0 and resp > 0 and not np.isnan(eff):
                valid_records.append((enrol, resp, eff))
        except ValueError:
            continue
    if valid_records:
        tot_resp = sum(x[1] for x in valid_records)
        weighted_eff = sum(x[1] * x[2] for x in valid_records) / tot_resp
        sis_eff_clean[code] = round(weighted_eff, 2)
        sis_enrol_clean[code] = sum(x[0] for x in valid_records) / len(valid_records)

        # estimate quota
        if max_enrol <= 80:
            sis_quota_est[code] = 80
        if max_enrol <= 120:
            sis_quota_est[code] = 120
        else:
            sis_quota_est[code] = 288

# ---------------------------------------------------------
# 4. Build Main Table Anchored on cc-site-info.json (Non-CCAI)
# ---------------------------------------------------------
print(f"len(cc_raw)={len(cc_raw)}")

rows = []
for code, info in sorted(cc_raw.items()):
    code = code.strip().upper()
    if code.startswith("CCAI"):
        continue

    course_name = info.get("course_name", "").strip()
    aoi = code[:4]

    is_online = 1 if ("online" in course_name.lower() or code[4] == "5") else 0
    is_comm_intensive = int(info.get("is_comm_intensive") is True)

    sem_str = info.get("sem_offered", "").lower()
    has_sem1 = "first" in sem_str or "sem 1" in sem_str
    has_sem2 = "second" in sem_str or "sem 2" in sem_str
    if has_sem1 and has_sem2:
        offered_sem = "both"
    elif has_sem1:
        offered_sem = "sem1"
    elif has_sem2:
        offered_sem = "sem2"
    else:
        offered_sem = "unknown"

    sdg_buckets = {f"sdg_{i:02d}": 0 for i in range(1, 18)}
    for sdg in info.get("sdgs", []):
        sdg_buckets[f"sdg_{int(sdg[-2:]):02d}"] = 1

    study_load_total = float(str(info.get("study_load_total", "0")).strip())
    load_buckets = categorize_study_load(info.get("study_load_dist", {}))
    ass_buckets = categorize_assessment(info.get("ass_dist", {}))

    # Enrolment Features (supports both sem1_quota/sem2_quota and fallback total_quota)
    enrol_info = enrol_raw.get(code, {})
    s1_q = enrol_info.get("sem1_quota", 0)
    s1_v = enrol_info.get("sem1_vacancies", 0)
    s2_q = enrol_info.get("sem2_quota", 0)
    s2_v = enrol_info.get("sem2_vacancies", 0)

    if (s1_q + s2_q) > 0:
        num_sems_active = (1 if s1_q > 0 else 0) + (1 if s2_q > 0 else 0)
        total_q = s1_q + s2_q
        total_v = s1_v + s2_v
        quota_per_semester = total_q / num_sems_active
        enrol_fill_rate = round((total_q - total_v) / total_q, 4)
        sem1_fill_rate = round((s1_q - s1_v) / s1_q, 4) if s1_q > 0 else np.nan
        sem2_fill_rate = round((s2_q - s2_v) / s2_q, 4) if s2_q > 0 else np.nan
    elif enrol_info.get("total_quota", 0) > 0:
        # Fallback if enrol-stat-info.json used old schema
        total_q = enrol_info["total_quota"]
        total_v = enrol_info.get("total_vacancies", 0)
        quota_per_semester = float(total_q)
        enrol_fill_rate = round((total_q - total_v) / total_q, 4)
        sem1_fill_rate = enrol_fill_rate if offered_sem in ["sem1", "both"] else np.nan
        sem2_fill_rate = enrol_fill_rate if offered_sem in ["sem2", "both"] else np.nan
    elif code in sis_quota_est:
        # Historical course not offered in 2026-27
        quota_per_semester = sis_quota_est[code]
        enrol_fill_rate = sis_enrol_clean[code] / quota_per_semester
        sem1_fill_rate = np.nan
        sem2_fill_rate = np.nan
    else:
        quota_per_semester = np.nan
        enrol_fill_rate = np.nan
        sem1_fill_rate = np.nan
        sem2_fill_rate = np.nan

    sis_mean_effectiveness = sis_eff_clean.get(code, np.nan)
    llm_info = llm_raw.get(code, {})
    llm_sentiment_score = llm_info.get("llm_sentiment_score", np.nan)
    llm_perceived_workload = llm_info.get("llm_workload_score", np.nan)

    rows.append({
        "course_code": code,
        "course_name": course_name,
        "aoi": aoi,
        "is_online": is_online,
        "is_comm_intensive": is_comm_intensive,
        "offered_sem": offered_sem,
        **sdg_buckets,
        "study_load_total": study_load_total,
        **load_buckets,
        **ass_buckets,
        "quota_per_semester": quota_per_semester,
        "sis_mean_effectiveness": sis_mean_effectiveness,
        "enrol_fill_rate": enrol_fill_rate,
        "sem1_fill_rate": sem1_fill_rate,
        "sem2_fill_rate": sem2_fill_rate,
        "llm_sentiment_score": llm_sentiment_score,
        "llm_perceived_workload": llm_perceived_workload
    })

df_main = pd.DataFrame(rows)

df_main = pd.get_dummies(df_main, columns=["aoi", "offered_sem"], prefix=["aoi", "sem"], dtype=int, drop_first=False)
df_main.insert(2, "aoi", df_main["course_code"].str[:4])

s1_pct = df_main["sem1_fill_rate"].rank(pct=True)
s2_pct = df_main["sem2_fill_rate"].rank(pct=True)
df_main["enrol_popularity_pct_sem_adj"] = np.nanmean(np.vstack([s1_pct, s2_pct]), axis=0).round(4)

print(f"Unmatched study_load keys ({len(unmatched_load_keys)}):", unmatched_load_keys)
print(f"Unmatched ass_dist keys ({len(unmatched_ass_keys)}):", unmatched_ass_keys)
# print("\n--- MISSING DATA SUMMARY (BEFORE MCAR IMPUTATION) ---")
# print(df_main[["quota_per_semester", "sis_mean_effectiveness", "enrol_fill_rate", "llm_sentiment_score", "llm_perceived_workload"]].isnull().sum())

df_main.to_csv("hku_cc_main_dataset_raw_na.csv", index=False)

len(cc_raw)=270
Unmatched study_load keys (0): set()
Unmatched ass_dist keys (0): set()


/tmp/ipykernel_35740/950049920.py:304: RuntimeWarning: Mean of empty slice
  df_main["enrol_popularity_pct_sem_adj"] = np.nanmean(np.vstack([s1_pct, s2_pct]), axis=0).round(4)


In [14]:
# Missingness report

miss = df_main.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_table = pd.DataFrame({
    "n_missing": miss,
    "pct_missing": (miss / len(df_main) * 100).round(1),
    "dtype": df_main[miss.index].dtypes.astype(str)
})

print("\n--- MISSING DATA SUMMARY (BEFORE MCAR IMPUTATION) ---")
print(miss_table)

df_main["n_missing"] = df_main.isnull().sum(axis=1)


--- MISSING DATA SUMMARY (BEFORE MCAR IMPUTATION) ---
                              n_missing  pct_missing    dtype
sem1_fill_rate                       74         27.4  float64
sem2_fill_rate                       69         25.6  float64
llm_sentiment_score                  22          8.1  float64
llm_perceived_workload               22          8.1  float64
enrol_fill_rate                       2          0.7  float64
sis_mean_effectiveness                2          0.7  float64
quota_per_semester                    2          0.7  float64
enrol_popularity_pct_sem_adj          2          0.7  float64


In [18]:
# Impute missing values

# MCAR: Median imputation
df_imputed = df_main.copy()
mcar = ["enrol_fill_rate", "sis_mean_effectiveness", "quota_per_semester", "enrol_popularity_pct_sem_adj"]
for col in mcar:
    if df_imputed[col].notnull().any():
        df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

# MAR: 

df_imputed.to_csv("hku_cc_main_dataset_imputed.csv", index=False)
print("\nSaved both 'hku_cc_main_dataset_raw_na.csv' and 'hku_cc_main_dataset_imputed.csv'!")

miss = df_imputed.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_table = pd.DataFrame({
    "n_missing": miss,
    "pct_missing": (miss / len(df_main) * 100).round(1),
    "dtype": df_main[miss.index].dtypes.astype(str)
})

print("\n--- MISSING DATA SUMMARY (AFTER MCAR IMPUTATION) ---")
print(miss_table)

df_main["n_missing"] = df_main.isnull().sum(axis=1)


Saved both 'hku_cc_main_dataset_raw_na.csv' and 'hku_cc_main_dataset_imputed.csv'!

--- MISSING DATA SUMMARY (AFTER MCAR IMPUTATION) ---
                        n_missing  pct_missing    dtype
sem1_fill_rate                 74         27.4  float64
sem2_fill_rate                 69         25.6  float64
llm_sentiment_score            22          8.1  float64
llm_perceived_workload         22          8.1  float64
